# R4 - Entrenamiento del segmentador baseline U-Net

El objetivo es entrenar un **U-Net baseline** para la delimitación automática de la región de interés en las imágenes ecográficas 2D, manteniendo la configuración experimental original:

- tamaño de entrada: 256×256;
- batch size: 4;
- optimizador: Adam;
- learning rate: 1e-4;
- función de pérdida: BCEWithLogitsLoss;
- umbral de binarización: 0.5;
- semilla: 42;
- selección del checkpoint: mayor Dice de validación.

## 1. Configuración del entorno

In [2]:
from pathlib import Path
import sys
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

PROJECT_ROOT = Path("../..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification
PyTorch: 2.5.1+cu121
CUDA de PyTorch: 12.1
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3080


## 2. Rutas del proyecto

In [3]:
DATASET_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "MMOTU_OTU_2D_experimental"
)

SPLIT_PATH = (
    PROJECT_ROOT /
    "configs" /
    "splits" /
    "mmotu_experimental_split_standard.csv"
)

MODEL_DIR = PROJECT_ROOT / "models" / "segmentation"
RESULT_DIR = PROJECT_ROOT / "results" / "segmentation" / "unet_baseline"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "unet_baseline_best.pth"
HISTORY_PATH = RESULT_DIR / "training_history.csv"

paths_to_check = {
    "dataset": DATASET_DIR,
    "split estándar": SPLIT_PATH,
    "dataset.py": PROJECT_ROOT / "src" / "segmentation" / "dataset.py",
    "unet.py": PROJECT_ROOT / "src" / "segmentation" / "unet.py",
}

for name, path in paths_to_check.items():
    print(f"{name:<18} {'OK' if path.exists() else 'NO ENCONTRADO'} -> {path}")

dataset            OK -> /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/data/processed/MMOTU_OTU_2D_experimental
split estándar     OK -> /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/configs/splits/mmotu_experimental_split_standard.csv
dataset.py         OK -> /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/segmentation/dataset.py
unet.py            OK -> /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification/src/segmentation/unet.py


## 3. Verificación de las particiones experimentales

In [4]:
for split in ["train", "validation", "test"]:
    images_dir = DATASET_DIR / split / "images"
    masks_dir = DATASET_DIR / split / "masks"
    print(
        f"{split:<12}",
        "images:", images_dir.exists(),
        "| masks:", masks_dir.exists(),
    )

split_df = pd.read_csv(SPLIT_PATH)
print("\nDistribución registrada en el manifiesto:")
print(split_df["experimental_split"].value_counts())

train        images: True | masks: True
validation   images: True | masks: True
test         images: True | masks: True

Distribución registrada en el manifiesto:
experimental_split
train         800
test          469
validation    200
Name: count, dtype: int64


## 4. Configuración baseline

Se mantiene la configuración utilizada en el experimento original para conservar la comparabilidad con los resultados obtenidos previamente.

In [5]:
from src.segmentation.dataset import OvarianTumorSegmentationDataset
from src.segmentation.unet import UNet

IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4

IN_CHANNELS = 3
OUT_CHANNELS = 1

LEARNING_RATE = 1e-4
THRESHOLD = 0.5
NUM_EPOCHS = 50
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3080


## 5. Construcción de Dataset y DataLoader

In [6]:
train_dataset = OvarianTumorSegmentationDataset(
    images_dir=DATASET_DIR / "train" / "images",
    masks_dir=DATASET_DIR / "train" / "masks",
    image_size=IMAGE_SIZE,
)

val_dataset = OvarianTumorSegmentationDataset(
    images_dir=DATASET_DIR / "validation" / "images",
    masks_dir=DATASET_DIR / "validation" / "masks",
    image_size=IMAGE_SIZE,
)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
    generator=train_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print("Train:", len(train_dataset), "imágenes |", len(train_loader), "batches")
print("Validation:", len(val_dataset), "imágenes |", len(val_loader), "batches")

Train: 800 imágenes | 200 batches
Validation: 200 imágenes | 50 batches


### Verificación de IDs

Se verifica que las imágenes físicas de `train` y `validation` correspondan con los IDs registrados en el manifiesto experimental estandarizado.

In [7]:
def assert_split_matches_manifest(dataset, split_name, split_df):
    manifest_ids = set(
        split_df.loc[
            split_df["experimental_split"] == split_name,
            "image_id",
        ].astype(str)
    )
    dataset_ids = set(str(image_id) for image_id in dataset.ids)

    print(
        split_name,
        "| manifiesto:", len(manifest_ids),
        "| dataset:", len(dataset_ids),
        "| faltantes:", len(manifest_ids - dataset_ids),
        "| adicionales:", len(dataset_ids - manifest_ids),
    )
    assert manifest_ids == dataset_ids

assert_split_matches_manifest(train_dataset, "train", split_df)
assert_split_matches_manifest(val_dataset, "validation", split_df)

print("\nParticiones verificadas correctamente.")

train | manifiesto: 800 | dataset: 800 | faltantes: 0 | adicionales: 0
validation | manifiesto: 200 | dataset: 200 | faltantes: 0 | adicionales: 0

Particiones verificadas correctamente.


## 6. Verificación del forward pass

In [8]:
batch = next(iter(train_loader))

images = batch["image"].to(device, non_blocking=True)
masks = batch["mask"].to(device, non_blocking=True)

model = UNet(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
).to(device)

model.eval()

with torch.no_grad():
    logits = model(images)

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Logits:", logits.shape)
print("Images device:", images.device)
print("Model device:", next(model.parameters()).device)

Images: torch.Size([4, 3, 256, 256])
Masks: torch.Size([4, 1, 256, 256])
Logits: torch.Size([4, 1, 256, 256])
Images device: cuda:0
Model device: cuda:0


## 7. Función de pérdida y métricas

In [9]:
criterion = nn.BCEWithLogitsLoss()

def dice_iou_scores(
    logits,
    targets,
    threshold=THRESHOLD,
    eps=1e-7,
):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    dimensions = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dimensions)
    pred_area = predictions.sum(dim=dimensions)
    target_area = targets.sum(dim=dimensions)

    dice = (2.0 * intersection + eps) / (pred_area + target_area + eps)
    union = pred_area + target_area - intersection
    iou = (intersection + eps) / (union + eps)

    return dice, iou

## 8. Funciones de entrenamiento y validación

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        batch_size = images.size(0)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            dice_values, iou_values = dice_iou_scores(logits, masks)

        total_loss += loss.item() * batch_size
        total_dice += dice_values.sum().item()
        total_iou += iou_values.sum().item()
        total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "dice": total_dice / total_samples,
        "iou": total_iou / total_samples,
    }


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    total_samples = 0

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            masks = batch["mask"].to(device, non_blocking=True)

            batch_size = images.size(0)
            logits = model(images)
            loss = criterion(logits, masks)
            dice_values, iou_values = dice_iou_scores(logits, masks)

            total_loss += loss.item() * batch_size
            total_dice += dice_values.sum().item()
            total_iou += iou_values.sum().item()
            total_samples += batch_size

    return {
        "loss": total_loss / total_samples,
        "dice": total_dice / total_samples,
        "iou": total_iou / total_samples,
    }

## 9. Entrenamiento definitivo del baseline

El checkpoint se selecciona mediante el mayor Dice obtenido en `validation`. El conjunto `test` permanece sin utilizar durante esta etapa.

In [11]:
# Reinicialización antes del entrenamiento definitivo
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = UNet(
    in_channels=IN_CHANNELS,
    out_channels=OUT_CHANNELS,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

history = []
best_val_dice = -1.0
best_epoch = None

training_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    val_metrics = validate_one_epoch(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    epoch_time = time.time() - epoch_start

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_dice": train_metrics["dice"],
            "train_iou": train_metrics["iou"],
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
            "epoch_seconds": epoch_time,
        }
    )

    if val_metrics["dice"] > best_val_dice:
        best_val_dice = val_metrics["dice"]
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_dice": val_metrics["dice"],
                "val_iou": val_metrics["iou"],
                "val_loss": val_metrics["loss"],
                "seed": SEED,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "threshold": THRESHOLD,
            },
            BEST_MODEL_PATH,
        )

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_metrics['loss']:.4f} "
        f"Dice: {train_metrics['dice']:.4f} "
        f"IoU: {train_metrics['iou']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} "
        f"Dice: {val_metrics['dice']:.4f} "
        f"IoU: {val_metrics['iou']:.4f} | "
        f"{epoch_time:.1f}s"
    )

training_time = time.time() - training_start

print("\n" + "=" * 60)
print("ENTRENAMIENTO FINALIZADO")
print("Mejor época:", best_epoch)
print(f"Mejor Dice de validación: {best_val_dice:.6f}")
print(f"Tiempo total: {training_time / 60:.2f} minutos")
print("\nCheckpoint:", BEST_MODEL_PATH)
print("Historial:", HISTORY_PATH)

Epoch [01/50] | Train Loss: 0.4394 Dice: 0.0000 IoU: 0.0000 | Val Loss: 0.4027 Dice: 0.0000 IoU: 0.0000 | 19.1s
Epoch [02/50] | Train Loss: 0.3280 Dice: 0.0681 IoU: 0.0517 | Val Loss: 0.2871 Dice: 0.4399 IoU: 0.3302 | 19.0s
Epoch [03/50] | Train Loss: 0.2614 Dice: 0.4161 IoU: 0.3129 | Val Loss: 0.2906 Dice: 0.5350 IoU: 0.4016 | 19.0s
Epoch [04/50] | Train Loss: 0.2490 Dice: 0.4851 IoU: 0.3707 | Val Loss: 0.2269 Dice: 0.5539 IoU: 0.4314 | 19.0s
Epoch [05/50] | Train Loss: 0.2310 Dice: 0.5233 IoU: 0.4067 | Val Loss: 0.2464 Dice: 0.4922 IoU: 0.3796 | 19.0s
Epoch [06/50] | Train Loss: 0.2249 Dice: 0.5320 IoU: 0.4165 | Val Loss: 0.2106 Dice: 0.5502 IoU: 0.4393 | 19.0s
Epoch [07/50] | Train Loss: 0.2158 Dice: 0.5629 IoU: 0.4474 | Val Loss: 0.2028 Dice: 0.5720 IoU: 0.4627 | 19.1s
Epoch [08/50] | Train Loss: 0.2047 Dice: 0.5705 IoU: 0.4569 | Val Loss: 0.1932 Dice: 0.5946 IoU: 0.4861 | 19.1s
Epoch [09/50] | Train Loss: 0.1977 Dice: 0.5889 IoU: 0.4743 | Val Loss: 0.2081 Dice: 0.6189 IoU: 0.4993 